# Fine-tune Parakeet for Speech Recognition

In this notebook, we fine-tune [NVIDIA Parakeet CTC](https://huggingface.co/nvidia/parakeet-ctc-1.1b) for automatic speech recognition. We cover both full fine-tuning and LoRA-based parameter-efficient training.

Parakeet is NVIDIA's CTC-based speech recognition model. Fine-tuning on domain-specific data can improve transcription accuracy for your use case.

**What we'll cover:**
- Loading and preprocessing an ASR dataset
- Setting up the Parakeet CTC data collator
- Full fine-tuning
- LoRA fine-tuning
- Running inference

In [ ]:
!pip install -q transformers datasets accelerate peft librosa soundfile jiwer

## Load Dataset

We use a small LibriSpeech subset for demonstration. Replace with your own dataset for real training — just ensure it has `audio` (16kHz) and `text` columns.

In [ ]:
from datasets import load_dataset, Audio

dataset = load_dataset("hf-internal-testing/librispeech_asr_dummy", "clean", split="validation")
dataset = dataset.cast_column("audio", Audio(sampling_rate=16000))

train_dataset = dataset
eval_dataset = dataset

print(f"Samples: {len(dataset)}")
print(dataset[0].keys())

## Load Model and Processor

In [ ]:
import torch
from transformers import AutoModelForCTC, AutoProcessor

model_checkpoint = "nvidia/parakeet-ctc-1.1b"

processor = AutoProcessor.from_pretrained(model_checkpoint)
model = AutoModelForCTC.from_pretrained(model_checkpoint)

print(f"Model: {model_checkpoint}")
print(f"Parameters: {sum(p.numel() for p in model.parameters()) / 1e6:.1f}M")

## Data Collator

The data collator processes raw audio and text through the Parakeet processor, which handles feature extraction and label tokenization for CTC training.

In [ ]:
class ParakeetDataCollator:
    def __init__(self, processor):
        self.processor = processor

    def __call__(self, features):
        texts = [f["text"] for f in features]
        audios = [f["audio"]["array"] for f in features]

        inputs = self.processor(
            audio=audios,
            text=texts,
            sampling_rate=self.processor.feature_extractor.sampling_rate,
        )
        return inputs

data_collator = ParakeetDataCollator(processor)

## Full Fine-tuning

In [ ]:
from transformers import TrainingArguments, Trainer

training_args = TrainingArguments(
    output_dir="./parakeet-finetuned",
    per_device_train_batch_size=2,
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=4,
    gradient_checkpointing=True,
    learning_rate=5e-5,
    max_steps=50,  # increase for real training
    bf16=True,
    logging_steps=10,
    eval_steps=50,
    save_steps=50,
    eval_strategy="steps",
    save_strategy="steps",
    report_to="none",
    remove_unused_columns=False,
    dataloader_num_workers=1,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    data_collator=data_collator,
)

trainer.train()

## LoRA Fine-tuning

Apply LoRA to the model's linear layers for parameter-efficient training. This is useful when GPU memory is limited.

In [ ]:
from peft import LoraConfig, get_peft_model

model_lora = AutoModelForCTC.from_pretrained(model_checkpoint)

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    target_modules=["linear"],
)

model_lora = get_peft_model(model_lora, lora_config)
model_lora.print_trainable_parameters()

In [ ]:
training_args_lora = TrainingArguments(
    output_dir="./parakeet-finetuned-lora",
    per_device_train_batch_size=2,
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=4,
    gradient_checkpointing=True,
    learning_rate=5e-5,
    max_steps=50,  # increase for real training
    bf16=True,
    logging_steps=10,
    eval_steps=50,
    save_steps=50,
    eval_strategy="steps",
    save_strategy="steps",
    report_to="none",
    remove_unused_columns=False,
    dataloader_num_workers=1,
)

trainer_lora = Trainer(
    model=model_lora,
    args=training_args_lora,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    data_collator=data_collator,
)

trainer_lora.train()

## Inference

In [ ]:
from transformers import pipeline

pipe = pipeline(
    "automatic-speech-recognition",
    model=model,
    feature_extractor=processor.feature_extractor,
    tokenizer=processor.tokenizer,
    device=0 if torch.cuda.is_available() else -1,
)

test_ds = load_dataset("hf-internal-testing/librispeech_asr_dummy", "clean", split="validation[:3]")
test_ds = test_ds.cast_column("audio", Audio(sampling_rate=16000))

for i, sample in enumerate(test_ds):
    result = pipe(sample["audio"]["array"])
    print(f"Sample {i+1}:")
    print(f"  Ground truth: {sample['text']}")
    print(f"  Prediction:   {result['text']}")
    print()